# NBA Team Stats Data
This is a notebook that will create a dataset that has all of the NBA teams and their stats from 2000 to 2025.

Source: [Basketball Reference](https://www.basketball-reference.com)

Authors: Rasheed Martin

In [1]:
from pathlib import Path
import polars as pl

## Combining Multiple CSVS into DataFrame

In this step we have already manually downloaded all the csvs from @ [Basketball Reference](https://www.basketball-reference.com). 
We had the option of webscrapping but we were having API issues so we pivoted. 

In [9]:
# Schema Definition
schema = {
    'Rk': pl.String, 'Player': pl.String, 'Age': pl.Float64, 'Team': pl.String, 'W': pl.Int64, 'L': pl.Int64, 'PW': pl.Float64, 'PL': pl.Float64, 'MOV': pl.Float64, 'SOS': pl.Float64, 'SRS': pl.Float64, 'ORtg': pl.Float64, 'DRtg': pl.Float64, 'NRtg': pl.String, 'Pace': pl.Float64, 'FTr': pl.Float64, '3PAr': pl.Float64, 'TS%': pl.Float64, 
    'eFG%': pl.Float64, 'TOV%': pl.Float64, "ORB%": pl.Float64 , "FT/FGA": pl.Float64, "DRB%": pl.Float64, "FT/FGA": pl.Float64, "Arena": pl.String, "Attend.": pl.Float64, "Attend./G": pl.Float64
}
df_list = []
# Loop over CSVs path and convert to individual dataframes
for file in Path("../data/raw_data/Team_Stats/").iterdir():
    df = pl.read_csv(file, skip_rows=1, truncate_ragged_lines=True)
    df = df.drop(["_duplicated_0", "_duplicated_1", ""])
    df = df.rename({
        "eFG%": "eFG%_Offense_Four_Factors",
        "TOV%": "TOV%_Offense_Four_Factors",
        "ORB%": "ORB%_Offense_Four_Factors",
        "FT/FGA": "FT/FGA_Offense_Four_Factors",
        "eFG%_duplicated_0": "eFG%_Defense_Four_Factors",
        "TOV%_duplicated_0": "TOV%_Defense_Four_Factors",
        "DRB%": "DRB%_Defense_Four_Factors",
        "FT/FGA_duplicated_0": "FT/FGA_Defense_Four_Factors",
        "Rk": "Rank",
        "W": "Win",
        "L": "Lose",
        "Attend.": "Total Attendance",
        "Attend./G": "Avg_Home_Game_Attendance"
    
    })
    name = file.stem.split("_")[-1]
    df = df.with_columns(
        pl.lit(name).alias("Season"),
        pl.col("Team").str.contains("*", literal=True).alias("Playoffs"),
        pl.col("Team").str.replace_all("*", "", literal=True),
    )
    df_list.append(df)
    
# Add the dataframes together vertically (more rows, matching columns) = More records
master_df = pl.concat(df_list, how="vertical")
master_df.head()

Rank,Team,Age,Win,Lose,PW,PL,MOV,SOS,SRS,ORtg,DRtg,NRtg,Pace,FTr,3PAr,TS%,eFG%_Offense_Four_Factors,TOV%_Offense_Four_Factors,ORB%_Offense_Four_Factors,FT/FGA_Offense_Four_Factors,eFG%_Defense_Four_Factors,TOV%_Defense_Four_Factors,DRB%_Defense_Four_Factors,FT/FGA_Defense_Four_Factors,Arena,Total Attendance,Avg_Home_Game_Attendance,Season,Playoffs
i64,str,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,i64,str,bool
1,"""Dallas Mavericks""",28.1,60,22,62,20,7.78,0.12,7.9,110.7,102.3,8.4,92.5,0.257,0.239,0.543,0.498,10.9,25.4,0.213,0.473,14.8,70.9,0.221,"""American Airlines Center""",816429,19913,"""2002-03""",true
2,"""Sacramento Kings""",28.3,59,23,59,23,6.5,0.18,6.68,105.9,99.1,6.8,95.4,0.262,0.184,0.535,0.499,13.3,25.6,0.196,0.446,13.6,70.6,0.204,"""ARCO Arena (II)""",709997,17317,"""2002-03""",true
3,"""San Antonio Spurs""",28.4,60,22,57,25,5.41,0.23,5.65,105.6,99.7,5.9,90.0,0.348,0.202,0.541,0.497,15.1,28.8,0.253,0.453,14.2,71.3,0.199,"""SBC Center""",735970,17950,"""2002-03""",true
4,"""New Jersey Nets""",27.3,49,33,56,26,5.22,-0.8,4.42,103.8,98.1,5.7,91.6,0.333,0.158,0.518,0.468,13.8,28.7,0.252,0.46,15.7,73.1,0.225,"""Continental Airlines Arena""",622574,15185,"""2002-03""",true
5,"""Detroit Pistons""",29.0,50,32,52,30,3.68,-0.71,2.97,104.1,99.9,4.2,86.8,0.322,0.237,0.523,0.473,13.3,25.9,0.248,0.461,13.9,74.0,0.225,"""The Palace of Auburn Hills""",839278,20470,"""2002-03""",true


In [11]:
master_df.null_count() # This makes sense

Rank,Team,Age,Win,Lose,PW,PL,MOV,SOS,SRS,ORtg,DRtg,NRtg,Pace,FTr,3PAr,TS%,eFG%_Offense_Four_Factors,TOV%_Offense_Four_Factors,ORB%_Offense_Four_Factors,FT/FGA_Offense_Four_Factors,eFG%_Defense_Four_Factors,TOV%_Defense_Four_Factors,DRB%_Defense_Four_Factors,FT/FGA_Defense_Four_Factors,Arena,Total Attendance,Avg_Home_Game_Attendance,Season,Playoffs
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
25,0,0,25,25,0,0,0,0,0,0,0,25,0,0,0,0,0,0,0,0,0,0,0,0,25,4,4,0,0


# Filter out the league averages and save to another table

In [15]:
league_average = master_df.filter(pl.col("Team") == "League Average")

In [16]:
league_average.null_count()

Rank,Team,Age,Win,Lose,PW,PL,MOV,SOS,SRS,ORtg,DRtg,NRtg,Pace,FTr,3PAr,TS%,eFG%_Offense_Four_Factors,TOV%_Offense_Four_Factors,ORB%_Offense_Four_Factors,FT/FGA_Offense_Four_Factors,eFG%_Defense_Four_Factors,TOV%_Defense_Four_Factors,DRB%_Defense_Four_Factors,FT/FGA_Defense_Four_Factors,Arena,Total Attendance,Avg_Home_Game_Attendance,Season,Playoffs
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
25,0,0,25,25,0,0,0,0,0,0,0,25,0,0,0,0,0,0,0,0,0,0,0,0,25,0,0,0,0


In [25]:
league_average = league_average.drop(
    [col for col in league_average.columns 
     if league_average.select(pl.col(col).is_null().all()).item()]
)

In [27]:
league_average.describe()

statistic,Team,Age,PW,PL,MOV,SOS,SRS,ORtg,DRtg,Pace,FTr,3PAr,TS%,eFG%_Offense_Four_Factors,TOV%_Offense_Four_Factors,ORB%_Offense_Four_Factors,FT/FGA_Offense_Four_Factors,eFG%_Defense_Four_Factors,TOV%_Defense_Four_Factors,DRB%_Defense_Four_Factors,FT/FGA_Defense_Four_Factors,Total Attendance,Avg_Home_Game_Attendance,Season,Playoffs
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64
"""count""","""25""",25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0,"""25""",25.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0
"""mean""",null,26.748,40.24,40.24,0.0,-0.0004,-0.0004,108.0,108.0,94.4,0.28288,0.27568,0.54532,0.50536,13.24,25.644,0.21548,0.50536,13.24,74.356,0.21548,679727.6,16870.04,null,0.0
"""std""",null,0.373184,2.146315,2.146315,0.0,0.002,0.002,3.585155,3.585155,3.527747,0.027233,0.083741,0.019676,0.02356,0.626498,2.185765,0.018581,0.02356,0.626498,2.185765,0.018581,138333.846184,3255.297061,null,null
"""min""","""League Average""",26.2,33.0,33.0,0.0,-0.01,-0.01,102.9,102.9,90.1,0.243,0.17,0.516,0.471,12.1,22.2,0.189,0.471,12.1,71.1,0.189,49476.0,1374.0,"""2000-01""",0.0
"""25%""",null,26.6,41.0,41.0,0.0,0.0,0.0,105.9,105.9,91.3,0.26,0.213,0.534,0.49,12.7,23.8,0.201,0.49,12.7,72.9,0.201,695899.0,17162.0,null,null
"""50%""",null,26.7,41.0,41.0,0.0,0.0,0.0,107.3,107.3,92.7,0.276,0.243,0.541,0.5,13.3,26.3,0.209,0.5,13.3,73.7,0.209,713186.0,17407.0,null,null
"""75%""",null,26.8,41.0,41.0,0.0,0.0,0.0,110.4,110.4,98.2,0.303,0.359,0.56,0.524,13.7,27.1,0.229,0.524,13.7,76.2,0.229,732148.0,17853.0,null,null
"""max""","""League Average""",27.8,41.0,41.0,0.0,0.0,0.0,115.3,115.3,100.3,0.333,0.421,0.581,0.547,14.2,28.9,0.248,0.547,14.2,77.8,0.248,751280.0,18330.0,"""2024-25""",0.0


In [28]:
league_average = league_average.drop(["Team", "MOV", "Playoffs"])

In [29]:
dirpath = Path("../data/clean_data/Team_Stats/")
dirpath.mkdir(parents=True, exist_ok=True)

if dirpath.exists():
    league_average.write_csv(dirpath / "league_average.csv", separator=",")

# Save off the original table without the league averages per season

In [31]:
master_df.null_count()

Rank,Team,Age,Win,Lose,PW,PL,MOV,SOS,SRS,ORtg,DRtg,NRtg,Pace,FTr,3PAr,TS%,eFG%_Offense_Four_Factors,TOV%_Offense_Four_Factors,ORB%_Offense_Four_Factors,FT/FGA_Offense_Four_Factors,eFG%_Defense_Four_Factors,TOV%_Defense_Four_Factors,DRB%_Defense_Four_Factors,FT/FGA_Defense_Four_Factors,Arena,Total Attendance,Avg_Home_Game_Attendance,Season,Playoffs
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
25,0,0,25,25,0,0,0,0,0,0,0,25,0,0,0,0,0,0,0,0,0,0,0,0,25,4,4,0,0


In [32]:
master_df = master_df.filter(pl.col("Team") != "League Average")
master_df

Rank,Team,Age,Win,Lose,PW,PL,MOV,SOS,SRS,ORtg,DRtg,NRtg,Pace,FTr,3PAr,TS%,eFG%_Offense_Four_Factors,TOV%_Offense_Four_Factors,ORB%_Offense_Four_Factors,FT/FGA_Offense_Four_Factors,eFG%_Defense_Four_Factors,TOV%_Defense_Four_Factors,DRB%_Defense_Four_Factors,FT/FGA_Defense_Four_Factors,Arena,Total Attendance,Avg_Home_Game_Attendance,Season,Playoffs
i64,str,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,i64,str,bool
1,"""Dallas Mavericks""",28.1,60,22,62,20,7.78,0.12,7.9,110.7,102.3,8.4,92.5,0.257,0.239,0.543,0.498,10.9,25.4,0.213,0.473,14.8,70.9,0.221,"""American Airlines Center""",816429,19913,"""2002-03""",true
2,"""Sacramento Kings""",28.3,59,23,59,23,6.5,0.18,6.68,105.9,99.1,6.8,95.4,0.262,0.184,0.535,0.499,13.3,25.6,0.196,0.446,13.6,70.6,0.204,"""ARCO Arena (II)""",709997,17317,"""2002-03""",true
3,"""San Antonio Spurs""",28.4,60,22,57,25,5.41,0.23,5.65,105.6,99.7,5.9,90.0,0.348,0.202,0.541,0.497,15.1,28.8,0.253,0.453,14.2,71.3,0.199,"""SBC Center""",735970,17950,"""2002-03""",true
4,"""New Jersey Nets""",27.3,49,33,56,26,5.22,-0.8,4.42,103.8,98.1,5.7,91.6,0.333,0.158,0.518,0.468,13.8,28.7,0.252,0.46,15.7,73.1,0.225,"""Continental Airlines Arena""",622574,15185,"""2002-03""",true
5,"""Detroit Pistons""",29.0,50,32,52,30,3.68,-0.71,2.97,104.1,99.9,4.2,86.8,0.322,0.237,0.523,0.473,13.3,25.9,0.248,0.461,13.9,74.0,0.225,"""The Palace of Auburn Hills""",839278,20470,"""2002-03""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
26,"""Milwaukee Bucks""",23.5,33,49,29,53,-4.18,0.2,-3.98,104.3,108.7,-4.4,94.2,0.276,0.189,0.537,0.499,14.2,24.9,0.207,0.51,14.2,73.1,0.221,"""BMO Harris Bradley Center""",621808,15166,"""2015-16""",false
27,"""Phoenix Suns""",26.0,23,59,24,58,-6.66,0.34,-6.32,102.2,109.0,-6.8,98.5,0.271,0.302,0.526,0.487,15.2,25.4,0.204,0.523,13.5,77.1,0.237,"""Talking Stick Resort Arena""",701405,17107,"""2015-16""",false
28,"""Brooklyn Nets""",26.9,21,61,22,60,-7.35,0.24,-7.12,103.2,110.9,-7.7,95.2,0.246,0.218,0.527,0.492,13.6,24.1,0.186,0.534,13.1,75.7,0.176,"""Barclays Center""",620142,15125,"""2015-16""",false


In [33]:
dirpath = Path("../data/clean_data/Team_Stats/")
dirpath.mkdir(parents=True, exist_ok=True)

if dirpath.exists():
    master_df.write_csv(dirpath / "master_df.csv", separator=",")

In [35]:
dirpath = Path("../data/clean_data/NBA_Player_Data/")

players_df = pl.read_csv(dirpath / "NBA_Player_Data_with_colleges.csv")
    

In [37]:
players_df.filter(pl.col("Team_Full_Name") == "Unknown")

Player,Age,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Season,MP_per_game,FG_per_game,FGA_per_game,3P_per_game,3PA_per_game,2P_per_game,2PA_per_game,…,PER,TS%,3PAr,FTr,ORB%,DRB%,TRB%,AST%,STL%,BLK%,TOV%,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,Team_Full_Name,Team_Abbreviation,MVP,DPOY,ROY,6MOY,CPOY,MIP,All_Star,DEF_Team,All_NBA,Height,Weight,Born,Birth_City,Birth_State,College
str,i64,str,i64,i64,i64,i64,i64,f64,i64,i64,f64,i64,i64,f64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str
"""Kemba Walker""",27,"""PG""",80,80,2736,588,1363,0.431,231,601,0.384,357,762,0.469,0.516,363,420,0.864,32,216,248,444,91,24,179,98,1770,0,"""2017-18""",34.2,7.4,17.0,2.9,7.5,4.5,9.5,…,20.6,0.572,0.441,0.308,1.2,8.7,4.9,27.4,1.6,0.7,10.4,27.4,6.5,2.0,8.6,0.15,4.7,-0.7,4.0,4.2,"""Unknown""","""CHO""",0,0,0,0,0,0,1,0,0,203,209,1964,"""Roberta""","""Georgia""","""University of Kentucky"""
"""Blake Griffin""",28,"""PF""",58,58,1970,436,996,0.438,111,322,0.345,325,674,0.482,0.493,259,330,0.785,73,355,428,334,41,18,165,139,1242,3,"""2017-18""",34.0,7.5,17.2,1.9,5.6,5.6,11.6,…,19.6,0.544,0.323,0.331,4.1,19.8,12.0,28.1,1.0,0.8,12.6,28.9,3.2,1.8,4.9,0.12,3.6,-0.2,3.4,2.7,"""Unknown""","""2TM""",0,0,0,0,0,0,0,0,0,208,249,1989,"""Oklahoma City""","""Oklahoma""","""University of Oklahoma"""
"""Tobias Harris""",25,"""PF""",80,80,2668,562,1221,0.46,183,445,0.411,379,776,0.488,0.535,179,216,0.829,58,379,437,192,71,35,105,164,1486,0,"""2017-18""",33.4,7.0,15.3,2.3,5.6,4.7,9.7,…,17.1,0.565,0.364,0.177,2.4,15.7,9.0,11.8,1.3,1.1,7.4,23.3,4.2,2.5,6.7,0.12,2.3,-0.4,1.9,2.6,"""Unknown""","""2TM""",0,0,0,0,0,0,0,0,0,190,190,1967,"""Monroe""","""Louisiana""","""University of New Orleans"""
"""Eric Bledsoe""",28,"""PG""",74,74,2322,468,989,0.473,125,360,0.347,343,629,0.545,0.536,252,317,0.795,51,233,284,370,149,42,216,185,1313,1,"""2017-18""",31.4,6.3,13.4,1.7,4.9,4.6,8.5,…,19.7,0.582,0.364,0.321,2.6,11.7,7.1,25.7,3.2,1.5,16.1,26.1,3.1,2.5,5.6,0.116,2.3,0.3,2.5,2.6,"""Unknown""","""2TM""",0,0,0,0,0,0,0,0,0,185,190,1989,"""Birmingham""","""Alabama""","""University of Kentucky"""
"""Dwight Howard""",32,"""C""",81,81,2463,506,911,0.555,1,7,0.143,505,904,0.559,0.556,334,582,0.574,255,757,1012,105,48,131,208,249,1347,0,"""2017-18""",30.4,6.2,11.2,0.0,0.1,6.2,11.2,…,20.5,0.577,0.008,0.639,11.0,33.8,22.2,7.1,1.0,4.4,15.1,24.2,2.9,3.9,6.8,0.132,-0.3,0.1,-0.1,1.2,"""Unknown""","""CHO""",0,0,0,0,0,0,0,0,0,211,265,1985,"""Atlanta""","""Georgia""","""Unknown"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Kirk Haston""",22,"""PF""",15,0,77,11,39,0.282,0,3,0.0,11,36,0.306,0.282,4,8,0.5,5,15,20,5,0,1,3,9,26,0,"""2001-02""",5.1,0.7,2.6,0.0,0.2,0.7,2.4,…,4.0,0.306,0.077,0.205,7.4,22.2,14.8,11.0,0.0,0.9,6.6,27.0,-0.3,0.1,-0.2,-0.116,-7.5,-3.8,-11.3,-0.2,"""Unknown""","""CHH""",0,0,0,0,0,0,0,0,0,206,240,1979,"""Loblesville""","""Tennessee""","""Indiana University"""
"""Charlie Bell""",22,"""SG""",7,0,44,3,11,0.273,0,6,0.0,3,5,0.6,0.273,2,2,1.0,2,3,5,2,0,0,5,3,8,0,"""2001-02""",6.3,0.4,1.6,0.0,0.9,0.4,0.7,…,-1.8,0.337,0.545,0.182,5.1,7.7,6.4,6.4,0.0,0.0,29.6,17.2,-0.2,0.0,-0.2,-0.172,-8.3,-4.0,-12.3,-0.1,"""Unknown""","""2TM""",0,0,0,0,0,0,0,0,0,190,198,1979,"""Flint""","""Michigan""","""Michigan State University"""
"""Dean Garrett""",35,"""C""",34,0,183,18,55,0.327,0,0,null,18,55,0.327,0.327,0,3,0.0,16,41,57,5,7,10,10,32,36,0,"""2001-02""",5.4,0.5,1.6,0.0,0.0,0.5,1.6,…,6.3,0.32,0.0,0.055,10.0,25.4,17.8,3.9,2.0,4.0,15.1,16.1,-0.4,0.3,-0.1,-0.024,-7.2,-0.2,-7.4,-0.3,"""Unknown""","""2TM""",0,0,0,0,0,0,0,0,0,208,225,1966,"""Los Angeles""","""California""","""Indiana University"""


In [38]:
players_df = players_df.with_columns(
    pl.when(pl.col("Team_Abbreviation") == "CHO")
    .then(pl.lit("Charlotte Hornets"))
    .otherwise(pl.col("Team_Full_Name"))
    .alias("Team_Full_Name")
)

In [40]:
players_df.write_csv(dirpath / "NBA_Player_Data_with_colleges.csv")

In [42]:
master_df

Rank,Team,Age,Win,Lose,PW,PL,MOV,SOS,SRS,ORtg,DRtg,NRtg,Pace,FTr,3PAr,TS%,eFG%_Offense_Four_Factors,TOV%_Offense_Four_Factors,ORB%_Offense_Four_Factors,FT/FGA_Offense_Four_Factors,eFG%_Defense_Four_Factors,TOV%_Defense_Four_Factors,DRB%_Defense_Four_Factors,FT/FGA_Defense_Four_Factors,Arena,Total Attendance,Avg_Home_Game_Attendance,Season,Playoffs
i64,str,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,i64,str,bool
1,"""Dallas Mavericks""",28.1,60,22,62,20,7.78,0.12,7.9,110.7,102.3,8.4,92.5,0.257,0.239,0.543,0.498,10.9,25.4,0.213,0.473,14.8,70.9,0.221,"""American Airlines Center""",816429,19913,"""2002-03""",true
2,"""Sacramento Kings""",28.3,59,23,59,23,6.5,0.18,6.68,105.9,99.1,6.8,95.4,0.262,0.184,0.535,0.499,13.3,25.6,0.196,0.446,13.6,70.6,0.204,"""ARCO Arena (II)""",709997,17317,"""2002-03""",true
3,"""San Antonio Spurs""",28.4,60,22,57,25,5.41,0.23,5.65,105.6,99.7,5.9,90.0,0.348,0.202,0.541,0.497,15.1,28.8,0.253,0.453,14.2,71.3,0.199,"""SBC Center""",735970,17950,"""2002-03""",true
4,"""New Jersey Nets""",27.3,49,33,56,26,5.22,-0.8,4.42,103.8,98.1,5.7,91.6,0.333,0.158,0.518,0.468,13.8,28.7,0.252,0.46,15.7,73.1,0.225,"""Continental Airlines Arena""",622574,15185,"""2002-03""",true
5,"""Detroit Pistons""",29.0,50,32,52,30,3.68,-0.71,2.97,104.1,99.9,4.2,86.8,0.322,0.237,0.523,0.473,13.3,25.9,0.248,0.461,13.9,74.0,0.225,"""The Palace of Auburn Hills""",839278,20470,"""2002-03""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
26,"""Milwaukee Bucks""",23.5,33,49,29,53,-4.18,0.2,-3.98,104.3,108.7,-4.4,94.2,0.276,0.189,0.537,0.499,14.2,24.9,0.207,0.51,14.2,73.1,0.221,"""BMO Harris Bradley Center""",621808,15166,"""2015-16""",false
27,"""Phoenix Suns""",26.0,23,59,24,58,-6.66,0.34,-6.32,102.2,109.0,-6.8,98.5,0.271,0.302,0.526,0.487,15.2,25.4,0.204,0.523,13.5,77.1,0.237,"""Talking Stick Resort Arena""",701405,17107,"""2015-16""",false
28,"""Brooklyn Nets""",26.9,21,61,22,60,-7.35,0.24,-7.12,103.2,110.9,-7.7,95.2,0.246,0.218,0.527,0.492,13.6,24.1,0.186,0.534,13.1,75.7,0.176,"""Barclays Center""",620142,15125,"""2015-16""",false


In [46]:
full_df = players_df.join(master_df, left_on=["Team_Full_Name", "Season"], right_on=["Team", "Season"], how="full", suffix="_team_stat")

In [48]:
full_df

Player,Age,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Season,MP_per_game,FG_per_game,FGA_per_game,3P_per_game,3PA_per_game,2P_per_game,2PA_per_game,…,All_NBA,Height,Weight,Born,Birth_City,Birth_State,College,Rank,Team,Age_team_stat,Win,Lose,PW,PL,MOV,SOS,SRS,ORtg,DRtg,NRtg,Pace,FTr_team_stat,3PAr_team_stat,TS%_team_stat,eFG%_Offense_Four_Factors,TOV%_Offense_Four_Factors,ORB%_Offense_Four_Factors,FT/FGA_Offense_Four_Factors,eFG%_Defense_Four_Factors,TOV%_Defense_Four_Factors,DRB%_Defense_Four_Factors,FT/FGA_Defense_Four_Factors,Arena,Total Attendance,Avg_Home_Game_Attendance,Season_team_stat,Playoffs
str,i64,str,i64,i64,i64,i64,i64,f64,i64,i64,f64,i64,i64,f64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,f64,f64,f64,f64,f64,f64,f64,…,i64,i64,i64,i64,str,str,str,i64,str,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,i64,str,bool
"""James Harden""",28,"""SG""",72,72,2551,651,1449,0.449,265,722,0.367,386,727,0.531,0.541,624,727,0.858,41,348,389,630,126,50,315,169,2191,4,"""2017-18""",35.4,9.0,20.1,3.7,10.0,5.4,10.1,…,1,196,218,1989,"""Los Angeles""","""California""","""Arizona State University""",1,"""Houston Rockets""",29.8,65,17,61,21,8.48,-0.27,8.21,114.7,106.1,8.6,97.6,0.298,0.502,0.59,0.551,12.7,21.3,0.233,0.521,13.4,79.9,0.171,"""Toyota Center""",732722,17871,"""2017-18""",true
"""Anthony Davis""",24,"""PF""",75,75,2727,780,1462,0.534,55,162,0.34,725,1300,0.558,0.552,495,598,0.828,187,645,832,174,115,193,162,159,2110,1,"""2017-18""",36.4,10.4,19.5,0.7,2.2,9.7,17.3,…,1,206,214,1968,"""Oakland""","""California""","""University of Texas at El Paso""",13,"""New Orleans Pelicans""",27.6,48,34,44,38,1.3,0.18,1.48,109.6,108.3,1.3,100.5,0.237,0.319,0.573,0.541,13.3,20.0,0.183,0.513,12.5,76.7,0.175,"""Smoothie King Center""",673920,16437,"""2017-18""",true
"""LeBron James""",33,"""PF""",82,82,3026,857,1580,0.542,149,406,0.367,708,1174,0.603,0.59,388,531,0.731,97,612,709,747,116,71,347,136,2251,18,"""2017-18""",36.9,10.5,19.3,1.8,5.0,8.6,14.3,…,1,203,249,1984,"""Akron""","""Ohio""","""Unknown""",14,"""Cleveland Cavaliers""",30.6,50,32,43,39,0.94,-0.35,0.59,112.9,111.9,1.0,98.0,0.275,0.379,0.584,0.547,12.6,20.1,0.214,0.54,12.2,77.3,0.166,"""Quicken Loans Arena""",843042,20562,"""2017-18""",true
"""Giannis Antetokounmpo""",23,"""PF""",75,75,2756,742,1402,0.529,43,140,0.307,699,1262,0.554,0.545,487,641,0.76,156,597,753,361,109,106,223,231,2014,1,"""2017-18""",36.7,9.9,18.7,0.6,1.9,9.3,16.8,…,2,211,220,1994,"""Athens""","""Greece""","""Unknown""",20,"""Milwaukee Bucks""",25.9,44,38,40,42,-0.3,-0.15,-0.45,109.8,110.1,-0.3,96.3,0.281,0.297,0.571,0.531,12.9,20.4,0.22,0.528,14.2,75.9,0.22,"""BMO Harris Bradley Center""",685303,16856,"""2017-18""",true
"""Damian Lillard""",27,"""PG""",73,73,2670,621,1415,0.439,227,629,0.361,394,786,0.501,0.519,493,538,0.916,62,263,325,481,77,27,206,117,1962,0,"""2017-18""",36.6,8.5,19.4,3.1,8.6,5.4,10.8,…,1,190,194,1990,"""Oakland""","""California""","""Weber State University""",9,"""Portland Trail Blazers""",25.6,49,33,48,34,2.6,0.0,2.6,109.1,106.4,2.7,96.5,0.24,0.324,0.549,0.511,12.3,23.3,0.192,0.503,11.7,79.1,0.193,"""Moda Center""",795750,19409,"""2017-18""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,16,"""Charlotte Bobcats""",25.5,43,39,40,42,-0.22,-0.67,-0.89,103.6,103.8,-0.2,92.4,0.297,0.219,0.522,0.481,11.7,21.9,0.219,0.491,12.6,77.6,0.183,"""Time Warner Cable Arena""",636268,15519,"""2013-14""",true
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,

PosixPath('../data/clean_data/NBA_Player_Data')

In [51]:
dirpath = Path('../data/clean_data/Team_Stats')
dirpath.mkdir(parents=True, exist_ok=True)
if dirpath.exists():
    full_df.write_csv(dirpath / "full_data.csv", separator=",")